In [3]:
from pathlib import Path
import polars as pl
import duckdb

In [4]:


parquet_candidates = [
    Path("../useful-views/psd_2026-09-24/psd_alldata.parquet"),
    Path("useful-views/psd_2026-09-24/psd_alldata.parquet"),
    Path("fas-psd/useful-views/psd_2026-09-24/psd_alldata.parquet"),
]
parquet_path = next((path.resolve() for path in parquet_candidates if path.is_file()), None)
if parquet_path is None:
    raise FileNotFoundError("Could not find fas-psd/useful-views/psd_2026-09-24/psd_alldata.parquet from this working directory.")

connection = duckdb.connect()
connection.read_parquet(str(parquet_path)).create_view("psd", replace=True)
connection.sql("DESCRIBE psd").pl()

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""Commodity_Code""","""VARCHAR""","""YES""",null,null,null
"""Commodity_Description""","""VARCHAR""","""YES""",null,null,null
"""Country_Code""","""VARCHAR""","""YES""",null,null,null
"""Country_Name""","""VARCHAR""","""YES""",null,null,null
"""Market_Year""","""SMALLINT""","""YES""",null,null,null
…,…,…,…,…,…
"""Attribute_ID""","""VARCHAR""","""YES""",null,null,null
"""Attribute_Description""","""VARCHAR""","""YES""",null,null,null
"""Unit_ID""","""VARCHAR""","""YES""",null,null,null


In [5]:
!pwd

/Users/jameshorine/repos/useful-views/fas-psd/sandbox


In [6]:
psdall = None
psdall = connection.sql(
    """SELECT * 
    FROM read_parquet('../useful-views/psd_2026-09-24/psd_alldata.parquet')
    LIMIT 100
    """).show()

┌────────────────┬────────────────────────┬──────────────┬──────────────┬─────────────┬───────────────┬───────┬──────────────┬────────────────────────┬─────────┬──────────────────┬────────┐
│ Commodity_Code │ Commodity_Description  │ Country_Code │ Country_Name │ Market_Year │ Calendar_Year │ Month │ Attribute_ID │ Attribute_Description  │ Unit_ID │ Unit_Description │ Value  │
│    varchar     │        varchar         │   varchar    │   varchar    │    int16    │     int16     │ int16 │   varchar    │        varchar         │ varchar │     varchar      │ double │
├────────────────┼────────────────────────┼──────────────┼──────────────┼─────────────┼───────────────┼───────┼──────────────┼────────────────────────┼─────────┼──────────────────┼────────┤
│ 0011000        │ Animal Numbers, Cattle │ AG           │ Algeria      │        1985 │          1985 │     0 │ 020          │ Beginning Stocks       │ 05      │ (1000 HEAD)      │ 1592.0 │
│ 0011000        │ Animal Numbers, Cattle │ AG    

In [7]:
pdspl = None
psdpl = pl.read_parquet('../useful-views/psd_2026-09-24/psd_alldata.parquet')

In [8]:
us_corn = psdpl.sql("""
    SELECT *
    FROM self
    WHERE Country_Name = 'United States'
      AND Commodity_Description = 'Corn'
      AND Attribute_Description = 'Area Harvested'
    ORDER BY Market_Year
""")

us_corn

Commodity_Code,Commodity_Description,Country_Code,Country_Name,Market_Year,Calendar_Year,Month,Attribute_ID,Attribute_Description,Unit_ID,Unit_Description,Value
str,str,str,str,i16,i16,i16,str,str,str,str,f64
"""0440000""","""Corn""","""US""","""United States""",1960,2006,7,"""004""","""Area Harvested""","""04""","""(1000 HA)""",28903.0
"""0440000""","""Corn""","""US""","""United States""",1961,2006,7,"""004""","""Area Harvested""","""04""","""(1000 HA)""",23324.0
"""0440000""","""Corn""","""US""","""United States""",1962,2006,7,"""004""","""Area Harvested""","""04""","""(1000 HA)""",22552.0
"""0440000""","""Corn""","""US""","""United States""",1963,2006,7,"""004""","""Area Harvested""","""04""","""(1000 HA)""",23968.0
"""0440000""","""Corn""","""US""","""United States""",1964,2006,7,"""004""","""Area Harvested""","""04""","""(1000 HA)""",22407.0
…,…,…,…,…,…,…,…,…,…,…,…
"""0440000""","""Corn""","""US""","""United States""",2022,2025,7,"""004""","""Area Harvested""","""04""","""(1000 HA)""",31851.0
"""0440000""","""Corn""","""US""","""United States""",2023,2026,1,"""004""","""Area Harvested""","""04""","""(1000 HA)""",35008.0
"""0440000""","""Corn""","""US""","""United States""",2024,2026,7,"""004""","""Area Harvested""","""04""","""(1000 HA)""",33608.0


In [9]:
psdpl.sql("""
    SELECT *
    FROM self
    WHERE Country_Name = 'United States'
      AND Commodity_Description = 'Corn'
      --AND Attribute_Description = 'Area Harvested'
      AND Market_Year = 2023
    ORDER BY Market_Year DESC
""").show(20)

Commodity_Code,Commodity_Description,Country_Code,Country_Name,Market_Year,Calendar_Year,Month,Attribute_ID,Attribute_Description,Unit_ID,Unit_Description,Value
str,str,str,str,i16,i16,i16,str,str,str,str,f64
"""0440000""","""Corn""","""US""","""United States""",2023,2026,1,"""004""","""Area Harvested""","""04""","""(1000 HA)""",35008.0
"""0440000""","""Corn""","""US""","""United States""",2023,2026,1,"""020""","""Beginning Stocks""","""08""","""(1000 MT)""",34551.0
"""0440000""","""Corn""","""US""","""United States""",2023,2026,1,"""028""","""Production""","""08""","""(1000 MT)""",389667.0
"""0440000""","""Corn""","""US""","""United States""",2023,2026,1,"""057""","""Imports""","""08""","""(1000 MT)""",722.0
"""0440000""","""Corn""","""US""","""United States""",2023,2026,1,"""081""","""TY Imports""","""08""","""(1000 MT)""",706.0
"""0440000""","""Corn""","""US""","""United States""",2023,2026,1,"""084""","""TY Imp. from U.S.""","""08""","""(1000 MT)""",0.0
"""0440000""","""Corn""","""US""","""United States""",2023,2026,1,"""086""","""Total Supply""","""08""","""(1000 MT)""",424940.0
"""0440000""","""Corn""","""US""","""United States""",2023,2026,1,"""088""","""Exports""","""08""","""(1000 MT)""",57275.0
"""0440000""","""Corn""","""US""","""United States""",2023,2026,1,"""113""","""TY Exports""","""08""","""(1000 MT)""",58520.0


In [25]:
psdpl.filter(
    ~pl.col("Attribute_ID").is_in(["181", "182", "183", "184", "195", "220"])  # drop rates/ratios
).group_by(
    "Commodity_Code", "Market_Year", "Attribute_ID"
).agg(
    pl.col("Commodity_Description", "Attribute_Description", "Unit_Description").first(),
    pl.col("Value").sum(),
    (pl.col("Value") != 0).sum().alias("n_countries"),
).sort("Commodity_Code", "Market_Year", "Attribute_ID")

Commodity_Code,Market_Year,Attribute_ID,Commodity_Description,Attribute_Description,Unit_Description,Value,n_countries
str,i16,str,str,str,str,f64,u32
"""0011000""",1960,"""020""","""Animal Numbers, Cattle""","""Beginning Stocks""","""(1000 HEAD)""",741042.0,53
"""0011000""",1960,"""023""","""Animal Numbers, Cattle""","""Dairy Cows Beg. Stocks""","""(1000 HEAD)""",82411.0,21
"""0011000""",1960,"""025""","""Animal Numbers, Cattle""","""Beef Cows Beg. Stocks""","""(1000 HEAD)""",54495.0,10
"""0011000""",1960,"""028""","""Animal Numbers, Cattle""","""Production""","""(1000 HEAD)""",61641.0,13
"""0011000""",1960,"""057""","""Animal Numbers, Cattle""","""Imports""","""(1000 HEAD)""",1107.0,9
…,…,…,…,…,…,…,…
"""4244000""",2026,"""140""","""Oil, Palm Kernel""","""Industrial Dom. Cons.""","""(1000 MT)""",6787.0,24
"""4244000""",2026,"""149""","""Oil, Palm Kernel""","""Food Use Dom. Cons.""","""(1000 MT)""",2067.0,25
"""4244000""",2026,"""161""","""Oil, Palm Kernel""","""Feed Waste Dom. Cons.""","""(1000 MT)""",6.0,1


In [26]:

world_view= None
world_view = psdpl.filter(
    ~pl.col("Attribute_ID").is_in(["181", "182", "183", "184", "195", "220"])
).group_by(
    "Commodity_Code", "Market_Year", "Attribute_ID"
).agg(
    pl.col("Commodity_Description", "Attribute_Description", "Unit_Description").first(),
    pl.col("Value").sum(),
    (pl.col("Value") != 0).sum().alias("n_countries"),
).sort("Commodity_Code", "Market_Year", "Attribute_ID")


In [30]:
world_view.filter(
    ~pl.col("Commodity_Description").is_in(["Corn"])
)

Commodity_Code,Market_Year,Attribute_ID,Commodity_Description,Attribute_Description,Unit_Description,Value,n_countries
str,i16,str,str,str,str,f64,u32
"""0011000""",1960,"""020""","""Animal Numbers, Cattle""","""Beginning Stocks""","""(1000 HEAD)""",741042.0,53
"""0011000""",1960,"""023""","""Animal Numbers, Cattle""","""Dairy Cows Beg. Stocks""","""(1000 HEAD)""",82411.0,21
"""0011000""",1960,"""025""","""Animal Numbers, Cattle""","""Beef Cows Beg. Stocks""","""(1000 HEAD)""",54495.0,10
"""0011000""",1960,"""028""","""Animal Numbers, Cattle""","""Production""","""(1000 HEAD)""",61641.0,13
"""0011000""",1960,"""057""","""Animal Numbers, Cattle""","""Imports""","""(1000 HEAD)""",1107.0,9
…,…,…,…,…,…,…,…
"""4244000""",2026,"""140""","""Oil, Palm Kernel""","""Industrial Dom. Cons.""","""(1000 MT)""",6787.0,24
"""4244000""",2026,"""149""","""Oil, Palm Kernel""","""Food Use Dom. Cons.""","""(1000 MT)""",2067.0,25
"""4244000""",2026,"""161""","""Oil, Palm Kernel""","""Feed Waste Dom. Cons.""","""(1000 MT)""",6.0,1


In [39]:
world_view.filter(
    pl.col("Commodity_Description") == "Corn"
    )


Commodity_Code,Market_Year,Attribute_ID,Commodity_Description,Attribute_Description,Unit_Description,Value,n_countries
str,i16,str,str,str,str,f64,u32
"""0440000""",1960,"""004""","""Corn""","""Area Harvested""","""(1000 HA)""",102179.0,76
"""0440000""",1960,"""020""","""Corn""","""Beginning Stocks""","""(1000 MT)""",56159.0,30
"""0440000""",1960,"""028""","""Corn""","""Production""","""(1000 MT)""",199576.0,76
"""0440000""",1960,"""057""","""Corn""","""Imports""","""(1000 MT)""",12822.0,45
"""0440000""",1960,"""081""","""Corn""","""TY Imports""","""(1000 MT)""",12765.0,45
…,…,…,…,…,…,…,…
"""0440000""",2026,"""125""","""Corn""","""Domestic Consumption""","""(1000 MT)""",1.313254e6,125
"""0440000""",2026,"""130""","""Corn""","""Feed Dom. Consumption""","""(1000 MT)""",827395.0,116
"""0440000""",2026,"""176""","""Corn""","""Ending Stocks""","""(1000 MT)""",272101.0,108


In [47]:
psdpl.select("Commodity_Code", 
             "Commodity_Description", 
             "Attribute_ID", 
             "Attribute_Description", 
             "Unit_Description").sort("Attribute_ID", "Commodity_Code").unique()

Commodity_Code,Commodity_Description,Attribute_ID,Attribute_Description,Unit_Description
str,str,str,str,str
"""4232000""","""Oil, Soybean""","""088""","""Exports""","""(1000 MT)"""
"""0813200""","""Meal, Peanut""","""125""","""Domestic Consumption""","""(1000 MT)"""
"""0571220""","""Tangerines/Mandarins, Fresh""","""028""","""Production""","""(1000 MT)"""
"""4232001""","""Oil, Soybean (Local)""","""140""","""Industrial Dom. Cons.""","""(1000 MT)"""
"""0813101""","""Meal, Soybean (Local)""","""086""","""Total Supply""","""(1000 MT)"""
…,…,…,…,…
"""0813500""","""Meal, Sunflowerseed""","""194""","""SME""","""(1000 MT)"""
"""0452000""","""Oats""","""130""","""Feed Dom. Consumption""","""(1000 MT)"""
"""0451000""","""Rye""","""020""","""Beginning Stocks""","""(1000 MT)"""


In [ ]:
psdpl.select("Commodity_Code", 
             "Commodity_Description", 
             "Attribute_ID", 
             "Attribute_Description", 
             "Unit_Description").sort("Attribute_ID", "Commodity_Code")



Commodity_Code,Commodity_Description,Attribute_ID,Attribute_Description,Unit_Description
str,str,str,str,str
"""0410000""","""Wheat""","""004""","""Area Harvested""","""(1000 HA)"""
"""0410000""","""Wheat""","""004""","""Area Harvested""","""(1000 HA)"""
"""0410000""","""Wheat""","""004""","""Area Harvested""","""(1000 HA)"""
"""0410000""","""Wheat""","""004""","""Area Harvested""","""(1000 HA)"""
"""0410000""","""Wheat""","""004""","""Area Harvested""","""(1000 HA)"""
…,…,…,…,…
"""0114200""","""Poultry, Meat, Broiler""","""220""","""Annual % Change Per Cap. Cons.""","""(PERCENT)"""
"""0114200""","""Poultry, Meat, Broiler""","""220""","""Annual % Change Per Cap. Cons.""","""(PERCENT)"""
"""0114200""","""Poultry, Meat, Broiler""","""220""","""Annual % Change Per Cap. Cons.""","""(PERCENT)"""


## Sample queries on the published views

The release folder `../useful-views/psd_2026-09-24/` has three views:

| View | One row per |
|---|---|
| `psd_alldata` | commodity × country × market year × attribute (the PSD file as published) |
| `commodity_attribute` | commodity × attribute: what PSD reports for each commodity, and in what unit |
| `world_balance` | commodity × market year × attribute: every country summed to a world total |

Every value is in the unit in `Unit_Description`, and units differ by commodity: never sum across units. Market years are local marketing years, and the latest years are USDA projections that change every month.

In [ ]:
VIEWS_DIR = Path("../useful-views/psd_2026-09-24")
PSD_ALLDATA = VIEWS_DIR / "psd_alldata.parquet"
COMMODITY_ATTRIBUTE = VIEWS_DIR / "commodity_attribute.parquet"
WORLD_BALANCE = VIEWS_DIR / "world_balance.parquet"

views = None
views = duckdb.connect()
for name, path in [("psd_alldata", PSD_ALLDATA), ("commodity_attribute", COMMODITY_ATTRIBUTE), ("world_balance", WORLD_BALANCE)]:
    views.read_parquet(str(path)).create_view(name, replace=True)

### What does PSD report for corn, and in what units?

`commodity_attribute` answers this without scanning two million rows. Corn's code is `0440000`.

In [ ]:
views.sql("""
    SELECT Attribute_ID, Attribute_Description, Unit_Description
    FROM commodity_attribute
    WHERE Commodity_Code = '0440000'
    ORDER BY Attribute_ID
""").pl()

### Which attributes change unit from one commodity to the next?

This is why you can't add up Production across commodities.

In [ ]:
views.sql("""
    SELECT Attribute_ID, Attribute_Description,
           count(DISTINCT Unit_Description) AS n_units,
           string_agg(DISTINCT Unit_Description, ' | ') AS units
    FROM commodity_attribute
    GROUP BY ALL
    HAVING count(DISTINCT Unit_Description) > 1
    ORDER BY Attribute_ID
""").pl()

### A country balance sheet: US corn

One row per attribute, one column per market year, the way USDA presents it. Supply (Beginning Stocks + Production + Imports = Total Supply) equals distribution (Exports + Domestic Consumption + Ending Stocks = Total Distribution).

In [ ]:
us_corn = None
us_corn = (
    pl.read_parquet(PSD_ALLDATA)
    .filter(
        pl.col("Country_Code") == "US",
        pl.col("Commodity_Code") == "0440000",
        pl.col("Market_Year") >= 2022,
    )
    .pivot(on="Market_Year", index=["Attribute_ID", "Attribute_Description", "Unit_Description"], values="Value")
    .sort("Attribute_ID")
)
us_corn

### The world balance sheet: corn over time

`world_balance` has the same columns as `psd_alldata`, without the country, plus `n_countries`. Here it's pivoted to one row per year.

In [ ]:
world_corn = None
world_corn = (
    pl.read_parquet(WORLD_BALANCE)
    .filter(pl.col("Commodity_Code") == "0440000", pl.col("Market_Year") >= 2016)
    .pivot(on="Attribute_Description", index="Market_Year", values="Value")
    .sort("Market_Year")
)
world_corn

### Top 10 corn producers and their share of the world

Joining a country row to its world row on (commodity, year, attribute).

In [ ]:
views.sql("""
    SELECT p.Country_Name,
           p.Value AS production_1000_mt,
           round(100 * p.Value / w.Value, 1) AS pct_of_world
    FROM psd_alldata p
    JOIN world_balance w USING (Commodity_Code, Market_Year, Attribute_ID)
    WHERE p.Commodity_Code = '0440000'
      AND p.Market_Year = 2025
      AND p.Attribute_ID = '028'  -- Production
    ORDER BY production_1000_mt DESC
    LIMIT 10
""").pl()

### World stocks-to-use, every commodity

Ending Stocks ÷ Domestic Consumption, as a percentage. At world level, exports and imports cancel out, so domestic consumption is total use. Sugar, cotton, livestock and fresh fruit report use or stocks under other attributes (or not at all), so they drop out here.

In [ ]:
views.sql("""
    SELECT Commodity_Description,
           sum(Value) FILTER (WHERE Attribute_ID = '176') AS ending_stocks,
           sum(Value) FILTER (WHERE Attribute_ID = '125') AS domestic_consumption,
           round(100 * ending_stocks / domestic_consumption, 1) AS stocks_to_use_pct
    FROM world_balance
    WHERE Market_Year = 2025
      AND Attribute_ID IN ('125', '176')
    GROUP BY ALL
    HAVING domestic_consumption > 0 AND ending_stocks IS NOT NULL
    ORDER BY stocks_to_use_pct DESC
""").pl()

### Stocks-to-use with and without China

China holds most of the world's corn stocks and much of its wheat, largely as state reserves that don't reach the world market. Analysts usually quote the world figure both ways. This subtracts China's rows in `psd_alldata` from `world_balance`.

In [ ]:
views.sql("""
    WITH china AS (
        SELECT Commodity_Code, Market_Year, Attribute_ID, Value
        FROM psd_alldata
        WHERE Country_Code = 'CH'
    )
    SELECT w.Commodity_Description,
           w.Market_Year,
           round(100 * sum(w.Value) FILTER (WHERE w.Attribute_ID = '176')
                     / sum(w.Value) FILTER (WHERE w.Attribute_ID = '125'), 1) AS world_pct,
           round(100 * sum(w.Value - coalesce(c.Value, 0)) FILTER (WHERE w.Attribute_ID = '176')
                     / sum(w.Value - coalesce(c.Value, 0)) FILTER (WHERE w.Attribute_ID = '125'), 1) AS ex_china_pct
    FROM world_balance w
    LEFT JOIN china c USING (Commodity_Code, Market_Year, Attribute_ID)
    WHERE w.Commodity_Code IN ('0410000', '0440000', '2222000')  -- wheat, corn, soybeans
      AND w.Market_Year >= 2020
      AND w.Attribute_ID IN ('125', '176')
    GROUP BY ALL
    ORDER BY w.Commodity_Description, w.Market_Year
""").pl()

### Biggest net exporters of soybeans

Exports − Imports per country. Change `DESC` to `ASC` for the biggest net importers.

In [ ]:
views.sql("""
    SELECT Country_Name,
           sum(Value) FILTER (WHERE Attribute_ID = '088') AS exports,
           sum(Value) FILTER (WHERE Attribute_ID = '057') AS imports,
           exports - imports AS net_exports
    FROM psd_alldata
    WHERE Commodity_Code = '2222000'
      AND Market_Year = 2025
      AND Attribute_ID IN ('057', '088')
    GROUP BY ALL
    ORDER BY net_exports DESC
    LIMIT 5
""").pl()

### More land or better yields? World corn by decade

World yield is world production ÷ world area. Averaging countries' yields would weight a small producer the same as the US. The 2020s include USDA projections.

In [ ]:
views.sql("""
    SELECT (Market_Year // 10) * 10 AS decade,
           round(avg(Value) FILTER (WHERE Attribute_ID = '004') / 1000, 1) AS area_million_ha,
           round(avg(Value) FILTER (WHERE Attribute_ID = '028') / 1000, 1) AS production_million_mt,
           round(sum(Value) FILTER (WHERE Attribute_ID = '028')
               / sum(Value) FILTER (WHERE Attribute_ID = '004'), 2) AS yield_mt_per_ha
    FROM world_balance
    WHERE Commodity_Code = '0440000'
      AND Attribute_ID IN ('004', '028')
    GROUP BY ALL
    ORDER BY decade
""").pl()